# Week 75, Day 1: SQL I: SELECT and WHERE

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w075d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shopdb.py', 'w', encoding='utf-8') as f:
    f.write('"""A small shop database for practising SQL (Week 75)."""\nimport sqlite3\n\nSCHEMA = """\nCREATE TABLE customers (\n    id     INTEGER PRIMARY KEY,\n    name   TEXT NOT NULL,\n    city   TEXT NOT NULL,\n    joined TEXT NOT NULL,\n    email  TEXT\n);\nCREATE TABLE products (\n    id       INTEGER PRIMARY KEY,\n    name     TEXT NOT NULL,\n    category TEXT NOT NULL,\n    price    REAL NOT NULL,\n    stock    INTEGER\n);\nCREATE TABLE orders (\n    id          INTEGER PRIMARY KEY,\n    customer_id INTEGER NOT NULL REFERENCES customers(id),\n    placed      TEXT NOT NULL,\n    status      TEXT NOT NULL\n);\nCREATE TABLE order_items (\n    order_id   INTEGER NOT NULL REFERENCES orders(id),\n    product_id INTEGER NOT NULL REFERENCES products(id),\n    qty        INTEGER NOT NULL,\n    unit_price REAL NOT NULL,\n    PRIMARY KEY (order_id, product_id)\n);\n"""\n\nDATA = """\nINSERT INTO customers VALUES\n    (1, \'Ada Lovelace\', \'London\', \'2024-01-15\', \'ada@example.com\'),\n    (2, \'Alan Turing\', \'Manchester\', \'2024-03-02\', \'alan@example.com\'),\n    (3, \'Grace Hopper\', \'New York\', \'2024-03-20\', NULL),\n    (4, \'Katherine Johnson\', \'Leeds\', \'2025-02-11\', \'kj@example.com\'),\n    (5, \'Tim Berners-Lee\', \'London\', \'2025-06-30\', NULL),\n    (6, \'Margaret Hamilton\', \'Bristol\', \'2025-09-09\', \'mh@example.com\');\nINSERT INTO products VALUES\n    (1, \'Tea\', \'drinks\', 3.50, 120),\n    (2, \'Coffee\', \'drinks\', 6.00, 80),\n    (3, \'Jam\', \'food\', 4.25, 0),\n    (4, \'Bread\', \'food\', 2.10, 35),\n    (5, \'Mug\', \'kitchen\', 8.00, 15),\n    (6, \'Teapot\', \'kitchen\', 24.00, 4),\n    (7, \'Biscuits\', \'food\', 1.80, 200),\n    (8, \'Hot chocolate\', \'drinks\', 5.50, NULL);\nINSERT INTO orders VALUES\n    (1, 1, \'2025-01-05\', \'delivered\'),\n    (2, 2, \'2025-01-17\', \'delivered\'),\n    (3, 1, \'2025-02-02\', \'cancelled\'),\n    (4, 3, \'2025-03-11\', \'delivered\'),\n    (5, 4, \'2025-03-29\', \'shipped\'),\n    (6, 1, \'2025-04-14\', \'delivered\'),\n    (7, 5, \'2025-05-01\', \'pending\'),\n    (8, 2, \'2025-05-20\', \'delivered\');\nINSERT INTO order_items VALUES\n    (1, 1, 2, 3.50), (1, 7, 3, 1.80),\n    (2, 2, 1, 6.00), (2, 5, 2, 8.00),\n    (3, 6, 1, 24.00),\n    (4, 1, 5, 3.25), (4, 3, 2, 4.25),\n    (5, 4, 1, 2.10), (5, 7, 2, 1.80), (5, 2, 1, 6.00),\n    (6, 6, 1, 22.00), (6, 5, 1, 8.00),\n    (7, 8, 2, 5.50),\n    (8, 1, 1, 3.50), (8, 4, 2, 2.10), (8, 7, 1, 1.80);\n"""\n\n\ndef connect():\n    """A fresh in-memory copy of the shop database."""\n    conn = sqlite3.connect(":memory:")\n    conn.executescript(SCHEMA + DATA)\n    return conn\n\n\ndef rows(sql, params=()):\n    """Run a query on a fresh copy of the database and return every row."""\n    conn = connect()\n    try:\n        return conn.execute(sql, params).fetchall()\n    finally:\n        conn.close()\n\n\ndef show(sql, params=()):\n    """Print a query\'s result as a small table."""\n    conn = connect()\n    cur = conn.execute(sql, params)\n    names = [d[0] for d in cur.description]\n    data = cur.fetchall()\n    widths = [max(len(str(x)) for x in [n] + [r[i] for r in data]) for i, n in enumerate(names)]\n    print("  ".join(n.ljust(w) for n, w in zip(names, widths)).rstrip())\n    for r in data:\n        print("  ".join(str(x).ljust(w) for x, w in zip(r, widths)).rstrip())\n    conn.close()\n')

**Predict before you run:** what will this print?



In [ ]:
from shopdb import show

show("""
SELECT name, price
FROM products
WHERE category = 'drinks' AND price < 6
ORDER BY price DESC
""")

In [ ]:
from shopdb import show

show("""
SELECT name,
       price * stock AS stock_value,
       CASE WHEN stock = 0 THEN 'sold out'
            WHEN stock < 20 THEN 'low'
            ELSE 'fine' END AS level
FROM products
WHERE stock IS NOT NULL
ORDER BY stock_value DESC
LIMIT 4
""")
print()
show("SELECT DISTINCT category FROM products ORDER BY category")

In [ ]:
from shopdb import show

show("""
SELECT name, city, joined
FROM customers
WHERE city IN ('London', 'Leeds')
   OR joined BETWEEN '2025-06-01' AND '2025-12-31'
ORDER BY joined
""")
print()
show("SELECT name FROM products WHERE name LIKE '%tea%' OR name LIKE 'B_ead'")

In [ ]:
from shopdb import rows

print(rows("SELECT count(*) FROM products WHERE stock <> 0"))
print(rows("SELECT count(*) FROM products WHERE stock <> 0 OR stock IS NULL"))
print(rows("SELECT NULL = NULL, NULL IS NULL, 1 + NULL, coalesce(NULL, 'default')"))
print(rows("SELECT name FROM customers WHERE email = NULL"), rows("SELECT name FROM customers WHERE email IS NULL"))

> **Key idea.** Tables have typed columns, rows without an order, and a primary key; constraints (NOT NULL, UNIQUE, CHECK, REFERENCES) protect the data. SELECT lists columns or expressions (with AS aliases and CASE), DISTINCT removes duplicate rows, ORDER BY sorts (DESC for highest first), and LIMIT and OFFSET page. Queries are evaluated FROM, WHERE, SELECT, DISTINCT, ORDER BY, LIMIT. WHERE combines comparisons with AND, OR (bracket them), NOT, IN, BETWEEN (inclusive) and LIKE (% and _). NULL is unknown: comparisons with it are unknown and filtered out, so test it with IS NULL, and use coalesce for defaults.

### Exercise: Choosing columns and order

Write three queries. `DRINKS`: the name and price of every drink, cheapest first. `LONDON`: the names of customers in London, in alphabetical order. `NEWEST`: the name and joining date of the **three** customers who joined most recently, newest first.

In [ ]:
DRINKS = ___
LONDON = ___
NEWEST = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_queries():\n    \"three SELECTs\"\n    from shopdb import rows\n    assert rows(DRINKS) == [(\"Tea\", 3.5), (\"Hot chocolate\", 5.5), (\"Coffee\", 6.0)]\n    assert rows(LONDON) == [(\"Ada Lovelace\",), (\"Tim Berners-Lee\",)]\n    assert rows(NEWEST) == [(\"Margaret Hamilton\", \"2025-09-09\"), (\"Tim Berners-Lee\", \"2025-06-30\"), (\"Katherine Johnson\", \"2025-02-11\")]")

### Exercise: Filtering rows

Write four queries, each ordered by name. `MID_PRICE`: the names of products priced from 3 to 8 inclusive that aren't in the kitchen category. `NO_EMAIL`: the names of customers with no email. `TEA`: the names of products whose name contains "tea" in any case. `CLEAR_OUT`: the name, category and stock of products in the food or kitchen categories with fewer than 20 in stock (leaving out unknown stock).

In [ ]:
MID_PRICE = ___
NO_EMAIL = ___
TEA = ___
CLEAR_OUT = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_filters():\n    \"WHERE in all its forms\"\n    from shopdb import rows\n    assert rows(MID_PRICE) == [(\"Coffee\",), (\"Hot chocolate\",), (\"Jam\",), (\"Tea\",)]\n    assert rows(NO_EMAIL) == [(\"Grace Hopper\",), (\"Tim Berners-Lee\",)]\n    assert rows(TEA) == [(\"Tea\",), (\"Teapot\",)]\n    assert rows(CLEAR_OUT) == [(\"Jam\", \"food\", 0), (\"Mug\", \"kitchen\", 15), (\"Teapot\", \"kitchen\", 4)]")

### Exercise: Computed columns

Write three queries. `VALUE`: each product's name and its stock value (`price * stock`, rounded to 2 places, named `value`), for products whose stock is known, highest value first. `LABELS`: each product's name and a column `label` that's `'sold out'` when stock is 0, `'unknown'` when it's NULL, `'low'` under 20 and `'ok'` otherwise, ordered by id. `WITH_VAT`: the name and price with 20% VAT added (rounded to 2 places), named `gross`, for the five cheapest products (cheapest first, by the price without VAT).

In [ ]:
VALUE = ___
LABELS = ___
WITH_VAT = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_computed():\n    \"expressions, CASE and aliases\"\n    from shopdb import rows, connect\n    assert rows(VALUE) == [(\"Coffee\", 480.0), (\"Tea\", 420.0), (\"Biscuits\", 360.0), (\"Mug\", 120.0), (\"Teapot\", 96.0), (\"Bread\", 73.5), (\"Jam\", 0.0)]\n    assert rows(LABELS) == [(\"Tea\", \"ok\"), (\"Coffee\", \"ok\"), (\"Jam\", \"sold out\"), (\"Bread\", \"ok\"), (\"Mug\", \"low\"), (\"Teapot\", \"low\"), (\"Biscuits\", \"ok\"), (\"Hot chocolate\", \"unknown\")]\n    assert rows(WITH_VAT) == [(\"Biscuits\", 2.16), (\"Bread\", 2.52), (\"Tea\", 4.2), (\"Jam\", 5.1), (\"Hot chocolate\", 6.6)]\n    names = [d[0] for d in connect().execute(VALUE).description]\n    assert names[1] == \"value\"")

### Exercise: Pages and distinct values

Write `page(n, size=3)`: a query (as a string) for page `n` (from 1) of the products listed by name, giving `id` and `name`, with `size` products per page; build it with an f-string of the two **integers** (convert them with `int()` first, so nothing else can get into the SQL). Then `CITIES`: the distinct cities customers live in, alphabetically, and `STATUSES`: the distinct order statuses, alphabetically.

In [ ]:
def page(n, size=3):
    ___

CITIES = ___
STATUSES = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_paging():\n    \"LIMIT, OFFSET and DISTINCT\"\n    from shopdb import rows\n    assert rows(page(1)) == [(7, \"Biscuits\"), (4, \"Bread\"), (2, \"Coffee\")]\n    assert rows(page(3)) == [(1, \"Tea\"), (6, \"Teapot\")] and rows(page(4)) == []\n    assert rows(page(2, size=5)) == [(5, \"Mug\"), (1, \"Tea\"), (6, \"Teapot\")]\n    assert rows(CITIES) == [(\"Bristol\",), (\"Leeds\",), (\"London\",), (\"Manchester\",), (\"New York\",)]\n    assert rows(STATUSES) == [(\"cancelled\",), (\"delivered\",), (\"pending\",), (\"shipped\",)]")

### Exercise: Counting with NULLs

Write three queries returning a single number each (with `count(*)`). `NOT_SOLD_OUT`: products whose stock isn't 0, **including** the one whose stock is unknown. `KNOWN_LOW`: products whose stock is known and under 50. `CONTACTABLE`: customers who have an email and aren't in London.

In [ ]:
NOT_SOLD_OUT = ___
KNOWN_LOW = ___
CONTACTABLE = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_nulls():\n    \"three-valued logic\"\n    from shopdb import rows\n    assert rows(NOT_SOLD_OUT) == [(7,)]\n    assert rows(KNOWN_LOW) == [(4,)]\n    assert rows(CONTACTABLE) == [(3,)]")

### Exercise: A table of your own

Write `SETUP`: SQL (several statements, run with `executescript`) that creates a table `suppliers` with columns `id` (an integer primary key), `name` (text, required, unique), `country` (text, required) and `rating` (an integer that must be between 1 and 5, checked with `CHECK`, and may be NULL), then inserts three suppliers: `Leaf & Co` from India rated 5, `Bean Brothers` from Colombia rated 4, and `Clayworks` from the UK with no rating. Then `TOP`: the names of suppliers rated 4 or more, best first.

In [ ]:
SETUP = """
CREATE TABLE suppliers (
    ___
);
"""
TOP = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_create():\n    \"CREATE TABLE, constraints and INSERT\"\n    import sqlite3\n    from shopdb import connect\n    conn = connect()\n    conn.executescript(SETUP)\n    assert conn.execute(\"SELECT count(*) FROM suppliers\").fetchone() == (3,)\n    assert conn.execute(TOP).fetchall() == [(\"Leaf & Co\",), (\"Bean Brothers\",)]\n    assert conn.execute(\"SELECT rating FROM suppliers WHERE name = 'Clayworks'\").fetchone() == (None,)\n    for bad in [\"INSERT INTO suppliers (name, country, rating) VALUES ('X', 'UK', 9)\",\n                \"INSERT INTO suppliers (name, country) VALUES ('Clayworks', 'UK')\",\n                \"INSERT INTO suppliers (name) VALUES ('Y')\"]:\n        try:\n            conn.execute(bad)\n            assert False, bad\n        except sqlite3.IntegrityError:\n            pass")

### Bug hunt: Equal to nothing

`email = NULL` is never true (it's unknown), so the query finds nobody and this prints `[]`. Test for NULL the right way, so it prints `[('Grace Hopper',), ('Tim Berners-Lee',)]`.

In [ ]:
from shopdb import rows

print(rows("SELECT name FROM customers WHERE email = NULL ORDER BY name"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"IS NULL\"\n    expect_output(\"[('Grace Hopper',), ('Tim Berners-Lee',)]\")")

### Bug hunt: AND before OR

The query should find cheap (under 4) food **or** drinks, but `AND` binds more tightly than `OR`, so it reads as "all food, or cheap drinks", and Jam (4.25) appears: this prints `['Biscuits', 'Bread', 'Jam', 'Tea']`. Add brackets so it prints `['Biscuits', 'Bread', 'Tea']`.

In [ ]:
from shopdb import rows

found = rows("SELECT name FROM products WHERE category = 'food' OR category = 'drinks' AND price < 4 ORDER BY name")
print([r[0] for r in found])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"brackets around OR\"\n    expect_output(\"['Biscuits', 'Bread', 'Tea']\")")

### Bug hunt: A pattern that only matched the end

The pattern `'%tea'` matches names that **end** with "tea", so `Teapot` is missed and this prints `['Tea']`. Change the pattern to match "tea" anywhere in the name, so it prints `['Tea', 'Teapot']`.

In [ ]:
from shopdb import rows

print([r[0] for r in rows("SELECT name FROM products WHERE name LIKE '%tea' ORDER BY name")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"anywhere in the name\"\n    expect_output(\"['Tea', 'Teapot']\")")

### Puzzle: The parts of a query

Arrange the clauses of a query for the two dearest products that are in stock, in the order SQL requires. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # rows("SELECT NULL = NULL")[0][0]
q = ___   # rows("SELECT count(*) FROM products WHERE price BETWEEN 2.10 AND 3.50")[0][0]
r = ___   # rows("SELECT 'O''Brien'")[0][0]
s = ___   # rows("SELECT 7 / 2, 7 / 2.0")[0]
t = ___   # the clause evaluated first in "SELECT ... FROM ... WHERE ... ORDER BY ...": "SELECT", "FROM", "WHERE" or "ORDER BY"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is None and q == 2 and r == \"O'Brien\", \"p, q and r\"\n    assert s == (3, 3.5) and t == \"FROM\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a safe query builder

Write `build_select(table, columns, where=None, order_by=None, limit=None)` that returns `(sql, params)` for `sqlite3`: values never go into the SQL text, only `?` placeholders, with the values in `params`. `table` and every column name must be a plain identifier (letters, digits and underscores, not starting with a digit) or `ValueError` is raised, since names can't be parameters. `where` is a dict of conditions joined with `AND`, in order: a key `"price"` means `price = ?`, and a key with a suffix after `__` chooses the operator: `__ne` (`<>`), `__lt`, `__le`, `__gt`, `__ge`, `__in` (a list: `price IN (?, ?, ?)`), `__like`, and `__isnull` (`True` gives `col IS NULL`, `False` gives `col IS NOT NULL`, with no parameter). `order_by` is a list of column names, each possibly prefixed with `-` for descending. `limit` must be a positive integer, added as `LIMIT ?`.

In [ ]:
import re

OPERATORS = {"ne": "<>", "lt": "<", "le": "<=", "gt": ">", "ge": ">=", "like": "LIKE"}

def build_select(table, columns, where=None, order_by=None, limit=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_builder():\n    \"placeholders and identifiers\"\n    from shopdb import rows\n    sql, params = build_select(\"products\", [\"name\", \"price\"], {\"category__in\": [\"food\", \"drinks\"], \"price__lt\": 4, \"stock__isnull\": False}, [\"-price\", \"name\"], 3)\n    assert sql == \"SELECT name, price FROM products WHERE category IN (?, ?) AND price < ? AND stock IS NOT NULL ORDER BY price DESC, name LIMIT ?\"\n    assert params == [\"food\", \"drinks\", 4, 3]\n    assert rows(sql, params) == [(\"Tea\", 3.5), (\"Bread\", 2.1), (\"Biscuits\", 1.8)]\n    sql, params = build_select(\"customers\", [\"name\"], {\"city\": \"London\", \"email__isnull\": True})\n    assert sql == \"SELECT name FROM customers WHERE city = ? AND email IS NULL\" and params == [\"London\"]\n    assert rows(sql, params) == [(\"Tim Berners-Lee\",)]\n    sql, params = build_select(\"customers\", [\"name\"], {\"name\": \"x' OR '1'='1\"})\n    assert rows(sql, params) == []\n    for bad in [(\"products; DROP TABLE products\", [\"name\"]), (\"products\", [\"name, price\"]), (\"products\", [\"1st\"])]:\n        try:\n            build_select(*bad)\n            assert False, bad\n        except ValueError:\n            pass\n    for bad_limit in (0, -1, \"3\", 2.5):\n        try:\n            build_select(\"products\", [\"name\"], limit=bad_limit)\n            assert False\n        except ValueError:\n            pass")

## Warm-up

Four lines of SQL say **what** you want (the names and prices of the drinks under 6, dearest first), and the database decides **how**. The page's `shopdb.py` (open it above) builds a small shop's database in memory: customers, products, orders and the items in each order. Everything today runs against it, with Python's built-in `sqlite3` doing the work. (Tomorrow's lesson covers `sqlite3` itself; today, `shopdb.show(sql)` prints a result and `shopdb.rows(sql)` returns it as a list of tuples.)

## Tables

A **relational database** keeps data in **tables**. Each table has named **columns**, each with a type, and holds any number of **rows**, one per thing: one row per customer, per product, per order. Rows have no built-in order; you ask for one when you read them.

```sql
CREATE TABLE products (
    id       INTEGER PRIMARY KEY,
    name     TEXT NOT NULL,
    category TEXT NOT NULL,
    price    REAL NOT NULL,
    stock    INTEGER
);
```

- Common types: `INTEGER`, `REAL` (floating point), `TEXT`, `BLOB` (bytes); bigger databases add `NUMERIC`/`DECIMAL` (exact, for money), `BOOLEAN`, `DATE` and `TIMESTAMP`. SQLite stores dates as `TEXT` in ISO format (`'2025-03-29'`), which sorts and compares correctly as text.
- The **primary key** identifies each row uniquely: here an integer `id`. In SQLite, `INTEGER PRIMARY KEY` is filled in automatically if you don't give one.
- **Constraints** protect the data: `NOT NULL` (a value is required), `UNIQUE`, `CHECK (price >= 0)`, and `REFERENCES other(id)`, a **foreign key** saying that a value must be the key of a row in another table (`orders.customer_id` refers to a customer).
- SQL keywords aren't case-sensitive (`select` works), but writing them in capitals is the common style. Strings use **single** quotes: `'London'`. A doubled quote inside one is a quote: `'O''Brien'`.

The shop's four tables: `customers(id, name, city, joined, email)`, `products(id, name, category, price, stock)`, `orders(id, customer_id, placed, status)` and `order_items(order_id, product_id, qty, unit_price)`.

## SELECT

```sql
SELECT columns FROM table WHERE condition ORDER BY columns LIMIT n OFFSET m;
```

- **`SELECT *`** returns every column; listing columns (`SELECT name, price`) returns just those, in that order. List them in real code: `*` breaks when columns are added.
- Columns can be **expressions**: `price * stock`, `upper(name)`, `round(price * 1.2, 2)`, `name || ' (' || category || ')'` (`||` joins strings). **`AS`** names a result column: `price * stock AS value`.
- **`DISTINCT`** removes duplicate result rows: `SELECT DISTINCT category FROM products`.
- **`ORDER BY`** sorts: `ORDER BY price DESC, name` sorts by price, highest first, then by name for ties. Without it, the order of rows isn't guaranteed, even if it looks stable.
- **`LIMIT n`** returns at most `n` rows; **`OFFSET m`** skips the first `m`, for pages of results.
- **`CASE WHEN condition THEN value ... ELSE value END`** is SQL's conditional expression.

The **logical order** in which a query is evaluated isn't the order it's written: `FROM` (which table), then `WHERE` (which rows), then `SELECT` (which columns and expressions), then `DISTINCT`, then `ORDER BY`, then `LIMIT`. That's why you can `ORDER BY` an alias defined in `SELECT`, but you can't use that alias in `WHERE`.

## WHERE

`WHERE` keeps the rows for which its condition is true:

- comparisons: `=` (one equals sign), `<>` or `!=`, `<`, `<=`, `>`, `>=`, on numbers, text and ISO dates;
- **`AND`**, **`OR`** and **`NOT`**, with `AND` binding more tightly than `OR` (as in Python), so use brackets when you mix them: `WHERE category = 'food' AND (price < 2 OR stock > 100)`;
- **`IN (a, b, c)`** for membership, and `NOT IN`;
- **`BETWEEN low AND high`**, which includes both ends;
- **`LIKE`** for patterns: `%` matches any run of characters (including none), `_` exactly one. `name LIKE 'T%'` starts with T; `'%tea%'` contains "tea". In SQLite, `LIKE` ignores the case of ASCII letters; in some databases it doesn't.

## NULL

**`NULL`** means "no value" or "unknown": Grace Hopper has no email; nobody knows how much hot chocolate is in stock. It behaves differently from every other value:

- Any comparison with `NULL` is neither true nor false but **unknown**: `NULL = NULL` is unknown, `stock > 10` is unknown when `stock` is `NULL`. `WHERE` keeps only rows where the condition is **true**, so unknown rows are dropped.
- Test for it with **`IS NULL`** and **`IS NOT NULL`**. `WHERE email = NULL` returns nothing, ever.
- `NOT unknown` is still unknown, which is why `WHERE stock <> 0` **doesn't** return the hot chocolate, and neither does `WHERE NOT (stock = 0)`.
- Arithmetic with `NULL` gives `NULL`; `coalesce(stock, 0)` replaces it with a default.
- `ORDER BY` puts `NULL`s first in SQLite (lowest), last in some other databases.

This **three-valued logic** (true, false, unknown) is the source of many subtle query bugs. When a column can be `NULL`, ask what your condition does for those rows.

## Practice

In these exercises you write SQL as Python strings. The checks run your queries on a fresh copy of the shop database and compare the rows they return, in order.

## On your own computer

SQLite comes with Python, and it also has a command-line shell, which is the quickest way to explore a database:

```bash
sqlite3 shop.db
```

Inside it, `.tables` lists the tables, `.schema products` shows how one was created, `.mode box` makes results readable, and any SQL statement ending in `;` runs. For a graphical view, the free **DB Browser for SQLite** opens the same files. When you move to a database server (PostgreSQL is the usual choice for Python web applications), almost everything from today works unchanged.

## Quiz

## Daily challenge

## Before you go

Write one question about the shop you'd like answered (who spends most? what sells best?) and try to write it in SQL. You'll probably need a **join** or **GROUP BY**, which is tomorrow: combining tables, and adding up groups of rows.